In [ ]:
import joblib
import re
import os
import glob
import json
import warnings
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, medfilt, sosfiltfilt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score, confusion_matrix
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score, recall_score
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {DEVICE}")

In [ ]:
CFG = {
    "target_fs": 100.0,       # Частота дискретизации после ресемплинга (Гц)
    "win_sec_det": 2.5,       # Длина окна для детектора движения (сек)
    "step_sec_det": 0.5,      # Шаг окна для детектора движения (сек)
    "win_sec_seg": 1.5,       # Длина окна для сегментатора фаз (сек)
    "step_sec_seg": 0.2,      # Шаг окна для сегментатора фаз (сек)
    "acc_cutoff": 8.0,       # Частота среза ФНЧ для акселерометра (Гц)
    "gyr_cutoff": 5.0,       # Частота среза ФНЧ для гироскопа (Гц)
    "epochs": 15,             # Количество эпох обучения
    "batch_size": 128,        # Размер батча
    "lr": 1e-3,               # Скорость обучения (Learning Rate)
}

WIN_SIZE_DET = int(CFG["target_fs"] * CFG["win_sec_det"])
STEP_SIZE_DET = int(CFG["target_fs"] * CFG["step_sec_det"])
WIN_SIZE_SEG = int(CFG["target_fs"] * CFG["win_sec_seg"])
STEP_SIZE_SEG = int(CFG["target_fs"] * CFG["step_sec_seg"])

LABELS_DIR = "/kaggle/input/datasets/daryushka/pd-signals-segmentation-5p/pd-signals-segmentation-5p/JSON_5_p"
SEGMENTS_DIR = "/kaggle/input/datasets/daryushka/pd-signals-segmentation-5p/pd-signals-segmentation-5p/extracted_segments"

def butter_lowpass(data, cutoff, fs, order=4):
    """Цифровой фильтр Баттерворта низких частот."""
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

def butter_highpass(data, cutoff, fs, order=2):
    """Высокочастотный фильтр для удаления дрейфа интегрирования."""
    nyq = 0.5 * fs
    sos = butter(order, cutoff / nyq, btype='high', analog=False, output='sos')
    return sosfiltfilt(sos, data)

def calculate_imu_angles(df):
    """Расчет угла ori_yaw при его отсутствии."""
    acc_x = df["acc_x"].values
    acc_y = df["acc_y"].values
    acc_z = df["acc_z"].values
    pitch = np.arctan2(acc_x, np.sqrt(acc_y**2 + acc_z**2))
    roll = np.arctan2(acc_y, acc_z)
    if "gyr_z" in df.columns:
        yaw = np.cumsum(df["gyr_z"].values * 1 / CFG['target_fs'])
    else:
        yaw = np.zeros_like(pitch)
        
    return yaw, pitch, roll

In [ ]:
def assign_detection_labels(t_center, json_data):
    """Генерация меток для детектора движения (0 - фон, 1 - тест)."""
    t1 = json_data["segment_1"]["start_time"]
    t5 = json_data["point_5"]["time_sec"]
    return 1 if (t_center >= t1 and t_center <= t5) else 0

def assign_walk_turn_labels(t_center, json_data, win_sec):

    t1 = json_data["segment_1"]["start_time"]
    t2 = json_data["segment_1"]["end_time"]
    t3 = json_data["segment_2"]["start_time"]
    t4 = json_data["segment_2"]["end_time"]
    t5 = json_data["point_5"]["time_sec"]
    
    half_win = win_sec / 2.0
    win_start = t_center - half_win
    win_end = t_center + half_win

    total_overlap_start = max(win_start, t1)
    total_overlap_end = min(win_end, t5)
    
    if (total_overlap_end - total_overlap_start) < half_win:
        return -1

    walk_time = 0.0
    o_start_1 = max(win_start, t1)
    o_end_1 = min(win_end, t2)
    if o_end_1 > o_start_1:
        walk_time += (o_end_1 - o_start_1)

    o_start_2 = max(win_start, t3)
    o_end_2 = min(win_end, t4)
    if o_end_2 > o_start_2:
        walk_time += (o_end_2 - o_start_2)

    return 0 if (walk_time >= (win_end - win_start) * 0.5) else 1

In [ ]:
def resample_and_prep(df, win_size=None, step_size=None, json_data=None):
    df = df.copy()
    needed_cols = ["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z"]
    for c in needed_cols:
        if c in df.columns:
            df[c] = df[c].ffill().bfill().fillna(0.0)

    yaw_calculated_from_gyro = False
    
    #if "ori_yaw" not in df.columns or df["ori_yaw"].isnull().all():
    #    df["ori_yaw"], _, _ = calculate_imu_angles(df)
    #    yaw_calculated_from_gyro = True  
    #else:
    #    df["ori_yaw"] = df["ori_yaw"].ffill().bfill().fillna(0.0)
    
    df["ori_yaw"], _, _ = calculate_imu_angles(df)
    yaw_calculated_from_gyro = True  
    
    t_old = df["time_sec"].values
    dt = 1.0 / CFG["target_fs"]
    t_new = np.arange(t_old[0], t_old[-1], dt)

    signal_cols = ["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z", "ori_yaw"]
    resampled = {"time_sec": t_new}
    for c in signal_cols:
        resampled[c] = np.interp(t_new, t_old, df[c].values)

    df_res = pd.DataFrame(resampled)

    for ax in ["x", "y", "z"]:
        df_res[f"acc_{ax}"] = butter_lowpass(df_res[f"acc_{ax}"].values, CFG["acc_cutoff"], CFG["target_fs"])
        df_res[f"gyr_{ax}"] = butter_lowpass(df_res[f"gyr_{ax}"].values, CFG["gyr_cutoff"], CFG["target_fs"])
    
    df_res["ori_yaw"] = butter_lowpass(df_res["ori_yaw"].values, CFG["gyr_cutoff"], CFG["target_fs"])

    if yaw_calculated_from_gyro:
        try:
            df_res["ori_yaw"] = butter_highpass(df_res["ori_yaw"].values, cutoff=0.1, fs=CFG["target_fs"])
        except ValueError:
            df_res["ori_yaw"] = detrend(df_res["ori_yaw"].values)
    
    df_res["acc_mag"] = np.sqrt(df_res["acc_x"]**2 + df_res["acc_y"]**2 + df_res["acc_z"]**2)
    df_res["gyr_mag"] = np.sqrt(df_res["gyr_x"]**2 + df_res["gyr_y"]**2 + df_res["gyr_z"]**2)

    cols_for_diff = ["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z", "ori_yaw", "acc_mag", "gyr_mag"]
    for c in cols_for_diff:
        raw_d1 = np.gradient(df_res[c].values, dt)
        df_res[f"{c}_d1"] = butter_lowpass(raw_d1, cutoff=3.0, fs=CFG["target_fs"])

    features = [
        "acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z", "ori_yaw",
        "acc_mag_d1", "gyr_mag_d1"
    ]

    data_mat = df_res[features].values
    data_mat = np.nan_to_num(data_mat)
    time_array = df_res["time_sec"].values

    return data_mat, time_array, df_res


def create_windows(data_mat, time_array, win_size, step_size, json_data=None, is_gait_segmenter=False):
    """
    Нарезка непрерывных данных на окна
    """
    X, T, labels = [], [], []
    current_win_sec = CFG["win_sec_seg"] if is_gait_segmenter else CFG["win_sec_det"]
    
    for i in range(0, len(data_mat) - win_size + 1, step_size):
        X.append(data_mat[i : i + win_size])
        t_c = time_array[i + win_size // 2]
        T.append(t_c)
        
        if json_data is not None:
            if is_gait_segmenter:
                labels.append(assign_walk_turn_labels(t_c, json_data, current_win_sec))
            else:
                labels.append(assign_detection_labels(t_c, json_data))
        else:
            labels.append(0)

    return np.array(X), np.array(T), np.array(labels)

In [ ]:
def extract_group_name(filename):
    upper_name = filename.upper()
    match_subject = re.search(r'\b(PD[0-9]+|H[0-9]+)\b', upper_name)
    if not match_subject:
        match_subject = re.search(r'(PD[0-9]+|H[0-9]+)', upper_name)
    if match_subject:
        subject_id = match_subject.group(1)
        remaining_part = upper_name[match_subject.end():]
        if "R1" in remaining_part: run_id = "r1"
        elif "R0" in remaining_part: run_id = "r0"
        else: run_id = "r1" if "R1" in upper_name else "r0"
    else:
        subject_id = "HEALTHY" if "HEALTHY" in upper_name else "UNKNOWN"
        run_id = "r1" if "R1" in upper_name else "r0"
    return f"{subject_id}_{run_id}"

In [ ]:
class AttentionLSTM(nn.Module):
    def __init__(self, input_dim, n_classes, hidden_dim=64, num_layers=2):
        super(AttentionLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            bidirectional=True
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )
        self.fc = nn.Linear(hidden_dim * 2, n_classes)

    def forward(self, x):
        # lstm_out: [batch_size, seq_len, hidden_dim * 2]
        lstm_out, _ = self.lstm(x)

        attn_weights = self.attention(lstm_out) # [batch_size, seq_len, 1]
        attn_weights = torch.softmax(attn_weights, dim=1)
 
        context = torch.sum(attn_weights * lstm_out, dim=1) # [batch_size, hidden_dim * 2]
        out = self.fc(context)
        return out

In [ ]:
def augment_time_series(X_chunk):
    X_aug = X_chunk.copy()
    seq_len, n_feats = X_aug.shape
    
    if np.random.rand() < 0.5:
        noise = np.random.normal(0, 0.01, size=(seq_len, 6))
        X_aug[:, :6] += noise
        
    if np.random.rand() < 0.5:
        alpha = np.random.uniform(-np.pi/12, np.pi/12)
        cos_a, sin_a = np.cos(alpha), np.sin(alpha)
        
        acc_y, acc_z = X_aug[:, 1].copy(), X_aug[:, 2].copy()
        X_aug[:, 1] = acc_y * cos_a - acc_z * sin_a
        X_aug[:, 2] = acc_y * sin_a + acc_z * cos_a

        if n_feats > 5:
            gyr_y, gyr_z = X_aug[:, 4].copy(), X_aug[:, 5].copy()
            X_aug[:, 4] = gyr_y * cos_a - gyr_z * sin_a
            X_aug[:, 5] = gyr_y * sin_a + gyr_z * cos_a
            
    if np.random.rand() < 0.5:
        X_aug *= np.random.uniform(0.9, 1.1)
        
    return X_aug

class SegDataset(Dataset):
    def __init__(self, X, y, is_training=False):

        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.is_training = is_training
        
        if len(self.X) == 0 or len(self.y) == 0:
            raise ValueError("Передан пустой массив данных X или y в SegDataset")
            
        unique_labels = np.unique(self.y)
        for label in unique_labels:
            if label < 0 or label > 1:
                raise ValueError(f"Критическая ошибка: обнаружен недопустимый класс {label} в данных разметки")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        X_chunk = self.X[idx]
        if self.is_training:
            X_chunk = augment_time_series(X_chunk)
            #X_chunk = X_chunk
        return torch.from_numpy(X_chunk.copy()).float(), torch.tensor(int(self.y[idx]), dtype=torch.long)

In [ ]:
def train_model(model, train_loader, val_loader, class_weights=None, is_detector=True):
    if class_weights is not None:
        weight_tensor = torch.FloatTensor(class_weights).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=weight_tensor)
    else:
        criterion = nn.CrossEntropyLoss()
        
    optimizer = optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-4)
    train_hist_loss, val_hist_loss = [], []
    
    best_bal_acc = 0.0  
    best_model_state = copy.deepcopy(model.state_dict())

    for epoch in range(CFG["epochs"]):
        model.train()
        tr_loss = 0.0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            tr_loss += loss.item() * batch_X.size(0)
        tr_loss /= len(train_loader.dataset)
        train_hist_loss.append(tr_loss)

        model.eval()
        all_preds, all_labels, vl_loss = [], [], 0.0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                vl_loss += loss.item() * batch_X.size(0)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(batch_y.cpu().numpy())
        
        current_bal_acc = balanced_accuracy_score(all_labels, all_preds)
        vl_loss /= len(val_loader.dataset)
        val_hist_loss.append(vl_loss)

        if current_bal_acc > best_bal_acc:
            best_bal_acc = current_bal_acc
            best_model_state = copy.deepcopy(model.state_dict()) 
            print(f"Epoch {epoch+1}: Новый лучший Balanced Accuracy: {best_bal_acc:.4f}")
        
    model.load_state_dict(best_model_state) 
    return model, train_hist_loss, val_hist_loss

In [ ]:
def smooth_predictions(preds, k=5):
    return medfilt(preds, kernel_size=k).astype(int)

def convert_windows_to_points(df, T_windows, window_seg_pred, det_pred, main_segment):
    """
    Проекция результатов классификации окон на временную шкалу исходных отсчетов.
    Принимает уже найденный main_segment
    """
    point_preds = np.zeros(len(df), dtype=int)
    t_points = df["time_sec"].values
    
    if main_segment is None:
        return point_preds
        
    st_t, en_t = T_windows[main_segment[0]], T_windows[main_segment[1] - 1]
    inside_det_mask = (t_points >= st_t) & (t_points <= en_t)

    closest_win_indices = np.searchsorted(T_windows, t_points[inside_det_mask])
    closest_win_indices = np.clip(closest_win_indices, 0, len(T_windows) - 1)
    
    prev_indices = np.clip(closest_win_indices - 1, 0, len(T_windows) - 1)
    diff_curr = np.abs(T_windows[closest_win_indices] - t_points[inside_det_mask])
    diff_prev = np.abs(T_windows[prev_indices] - t_points[inside_det_mask])
    
    final_indices = np.where(diff_prev < diff_curr, prev_indices, closest_win_indices)
    point_preds[inside_det_mask] = window_seg_pred[final_indices]
        
    return point_preds

def calculate_and_format_metrics(y_true, y_pred, title="", print_table=False):
    if len(y_true) == 0 or len(y_pred) == 0:
        metrics = {"Macro F1": 0.0, "Balanced F1": 0.0, "Balanced Accuracy": 0.0,
                   "Macro Recall": 0.0, "Sensitivity": 0.0, "Specificity": 0.0, "Precision_1": 0.0}
        if print_table: print(f"\n{title}\nНет данных для оценки.")
        return metrics

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = len(y_true) - np.sum(y_pred), 0, 0, np.sum(y_pred)

    sensitivity = tp / (tp + fn + 1e-6)
    specificity = tn / (tn + fp + 1e-6)
    precision_1 = tp / (tp + fp + 1e-6)
    bal_f1 = 2 * (precision_1 * sensitivity) / (precision_1 + sensitivity + 1e-6)

    metrics = {"Macro F1": macro_f1, "Balanced F1": bal_f1, "Balanced Accuracy": bal_acc,
               "Macro Recall": macro_recall, "Sensitivity": sensitivity, "Specificity": specificity, "Precision_1": precision_1}

    if print_table and title:
        print(f"\n{'='*60}\n {title}\n{'='*60}")
        print(pd.DataFrame([metrics]).T.round(4).to_string(header=False))
        print(f"\nМатрица ошибок:\n{pd.DataFrame(cm, index=['Истина 0', 'Истина 1'], columns=['Предсказано 0', 'Предсказано 1']).to_string()}")
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=['0', '1'], yticklabels=['Истина 0', 'Истина 1'])
        axes[0].set_title('Абсолютные значения')
        
        cm_norm = np.nan_to_num(cm.astype('float') / cm.sum(axis=1, keepdims=True))
        sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', ax=axes[1], xticklabels=['0', '1'], yticklabels=['Истина 0', 'Истина 1'], vmin=0, vmax=1)
        axes[1].set_title('Нормированная (по строкам)')
        
        plt.tight_layout()
        plt.show()
        print(f"{'='*60}\n")

    return metrics

def plot_loss_curves(tr_loss, va_loss, fold_num, model_name):
    """Отрисовка кривых обучения для фолда."""
    plt.figure(figsize=(6, 3))
    plt.plot(tr_loss, label='Train Loss')
    plt.plot(va_loss, label='Val Loss')
    plt.title(f'Fold {fold_num} - {model_name} Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

def extract_precise_timestamps(time_signals, point_seg_pred, det_pred, pred_segment, T_det):

    timestamps = {
        "T1_start": None,
        "T2_turn1_start": None,
        "T3_walk2_start": None,
        "T4_turn2_start": None,
        "T5_end": None
    }
    
    inside_mov = np.where(det_pred == 1)[0]
    if len(inside_mov) == 0 or pred_segment is None:
        return timestamps

    t_start_mov = T_det[pred_segment[0]]
    t_end_mov = T_det[pred_segment[1]-1]
    
    idx_start_df = np.argmin(np.abs(time_signals - t_start_mov))
    idx_end_df = np.argmin(np.abs(time_signals - t_end_mov))
    
    if idx_end_df <= idx_start_df:
        return timestamps

    timestamps["T1_start"] = float(time_signals[idx_start_df])
    timestamps["T5_end"] = float(time_signals[idx_end_df])

    sub_seg = point_seg_pred[idx_start_df:idx_end_df+1]
    sub_time = time_signals[idx_start_df:idx_end_df+1]
    
    transitions = []
    for i in range(1, len(sub_seg)):
        if sub_seg[i] != sub_seg[i-1]:
            transitions.append((sub_time[i], sub_seg[i]))
            
    t2_val, t3_val, t4_val = None, None, None
    
    for t, cls in transitions:
        if cls == 1 and t2_val is None:
            t2_val = float(t)
            continue
        if cls == 0 and t2_val is not None and t3_val is None:
            t3_val = float(t)
            continue
        if cls == 1 and t3_val is not None and t4_val is None:
            t4_val = float(t)
            break
            
    timestamps["T2_turn1_start"] = t2_val
    timestamps["T3_walk2_start"] = t3_val
    timestamps["T4_turn2_start"] = t4_val
    
    return timestamps

def plot_results_v3(df, T, det_pred, point_seg_pred, json_data, timestamps, title, save_json_name=None):
    acc_mag = np.sqrt(df["acc_x"]**2 + df["acc_y"]**2 + df["acc_z"]**2)
    time_signals = df["time_sec"].values

    fig, ax = plt.subplots(6, 1, figsize=(22, 18), sharex=True)

    ax[0].plot(df["time_sec"], df["acc_x"], alpha=0.5, label="acc_x", color="tab:blue")
    ax[0].plot(df["time_sec"], df["acc_y"], alpha=0.5, label="acc_y", color="tab:cyan")
    ax[0].plot(df["time_sec"], df["acc_z"], alpha=0.5, label="acc_z", color="navy")
    ax[0].plot(df["time_sec"], df["gyr_x"], alpha=0.5, label="gyr_x", color="tab:red")
    ax[0].plot(df["time_sec"], df["gyr_y"], alpha=0.5, label="gyr_y", color="tab:orange")
    ax[0].plot(df["time_sec"], df["gyr_z"], alpha=0.5, label="gyr_z", color="brown")
    ax[0].plot(df["time_sec"], df["ori_yaw"], alpha=0.8, label="ori_yaw", color="purple")
    ax[0].set_title(f"{title} | Полный набор сигналов")
    ax[0].grid(True); ax[0].legend(loc="upper right", ncol=4)

    if json_data is not None:
        try:
            gt_t1 = json_data["segment_1"]["start_time"]
            gt_t2 = json_data["segment_1"]["end_time"]
            gt_t3 = json_data["segment_2"]["start_time"]
            gt_t5 = json_data["point_5"]["time_sec"]
            gt_t4 = gt_t5 - json_data["point_5"]["turn2_duration_sec"]

            ax[1].axvspan(gt_t1, gt_t5, color='lime', alpha=0.4)
            ax[3].axvspan(gt_t1, gt_t2, color='lime', alpha=0.5)
            ax[3].axvspan(gt_t2, gt_t3, color='red', alpha=0.5)
            ax[3].axvspan(gt_t3, gt_t4, color='lime', alpha=0.5)
            ax[3].axvspan(gt_t4, gt_t5, color='red', alpha=0.5)
        except KeyError:
            pass
    
    ax[1].set_title("GROUND TRUTH ОБЩЕЕ ДВИЖЕНИЕ")
    ax[1].set_ylim(0, 1); ax[1].grid(True)
    ax[3].set_title("GROUND TRUTH ЭТАЛОННАЯ РАЗМЕТКА СЕГМЕНТОВ")
    ax[3].set_ylim(0, 1); ax[3].grid(True)

    pred_segment = find_true_movement_boundaries(det_pred, fs_windows=None, min_duration_sec=1.5)
    
    if pred_segment is not None:
        ax[2].axvspan(T[pred_segment[0]], T[pred_segment[1]-1], color='lime', alpha=0.4)
        
    ax[2].set_title("PREDICTED ОБЩЕЕ ДВИЖЕНИЕ (ДЕТЕКТОР)")
    ax[2].set_ylim(0, 1); ax[2].grid(True)

    if pred_segment is not None:
        idx_start_df = np.argmin(np.abs(time_signals - T[pred_segment[0]]))
        idx_end_df = np.argmin(np.abs(time_signals - T[pred_segment[1]-1]))
        
        if idx_end_df > idx_start_df:
            start_p = idx_start_df
            curr_cls = point_seg_pred[start_p]
            for idx in range(idx_start_df + 1, idx_end_df + 1):
                cls = point_seg_pred[idx]
                if cls != curr_cls:
                    ax[4].axvspan(time_signals[start_p], time_signals[idx-1], color="lime" if curr_cls == 0 else "red", alpha=0.5)
                    start_p = idx
                    curr_cls = cls
            ax[4].axvspan(time_signals[start_p], time_signals[idx_end_df], color="lime" if curr_cls == 0 else "red", alpha=0.5)

    ax[4].set_title("PREDICTED ПОТОЧЕЧНАЯ РАЗМЕТКА (Зеленый = Шаг, Красный = Поворот)")
    ax[4].set_ylim(0, 1); ax[4].grid(True)

    ax[5].plot(df["time_sec"], acc_mag, lw=1.2, color="gray", alpha=0.6, label="|ACC|")
    
    if timestamps:
        colors_t = {"T1_start": "forestgreen", "T2_turn1_start": "crimson", "T3_walk2_start": "darkorange", "T4_turn2_start": "red", "T5_end": "navy"}
        for key, value in timestamps.items():
            if value is not None and not np.isnan(value):
                ax[5].axvline(x=value, color=colors_t[key], linestyle="--", lw=2, label=f"Вычисленная {key}")
        
        if save_json_name:
            with open(save_json_name, "w") as fj:
                json.dump(timestamps, fj, indent=4)

    ax[5].set_title("ИТОГОВОЕ ПОЛОЖЕНИЕ НАЙДЕННЫХ ТОЧЕК НА СИГНАЛЕ ACC")
    ax[5].grid(True); ax[5].legend(loc="upper right", ncol=5)

    plt.xlabel("Время (сек)")
    plt.tight_layout()
    plt.show()

In [ ]:
def find_true_movement_boundaries(det_pred, fs_windows, min_duration_sec=2):
    """
    Находит изолированные блоки активности детектора и удаляет те, 
    которые длятся меньше min_duration_sec.
    Возвращает (start_idx, end_idx) для самого длинного/основного блока.
    """
    step_sec = CFG["step_sec_det"] 
    min_windows = int(min_duration_sec / step_sec)

    padded = np.clip(det_pred, 0, 1)
    padded = np.concatenate(([0], padded, [0]))
    diff = np.diff(padded)
    
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    
    valid_segments = []
    
    for s, e in zip(starts, ends):
        duration_windows = e - s
        if duration_windows >= min_windows:
            valid_segments.append((s, e))
            
    if len(valid_segments) == 0:
        return None 
        

    main_seg = max(valid_segments, key=lambda x: x[1] - x[0])
    return main_seg

In [ ]:
if __name__ == "__main__":
    N_FEATURES = 9 
    
    dataset = []
    jsons = glob.glob(os.path.join(LABELS_DIR, "**/*.json"), recursive=True)
    
    for jf in jsons:
        with open(jf) as f: data = json.load(f)
        if "segment_1" not in data or "segment_2" not in data or "point_5" not in data: continue
        base_name = os.path.basename(jf).replace("_segments.json", "")
        xlsx = glob.glob(os.path.join(SEGMENTS_DIR, f"*{base_name}*.xlsx"))
        if len(xlsx) > 0:
            df = pd.read_excel(xlsx[0])
            data_mat, time_array, df_proc = resample_and_prep(df) 
            dataset.append({"name": base_name, "group_name": extract_group_name(base_name),
                            "data_mat": data_mat, "time_array": time_array, "df_proc": df_proc, "raw_json": data})
            
    print(f"Загружено и предобработано {len(dataset)} файлов.")
    if len(dataset) == 0: raise RuntimeError("Не удалось загрузить ни одного файла!")

    groups_by_file = np.array([d["group_name"] for d in dataset])
    logo = LeaveOneGroupOut(); dummy_y = np.zeros(len(dataset)) 
    statistical_records, fold_metrics = [], []
    all_det_true, all_det_pred = [], []
    all_seg_true, all_seg_pred = [], []

    for fold, (tr_file_idx, va_file_idx) in enumerate(logo.split(dummy_y, dummy_y, groups_by_file)):
        test_group = groups_by_file[va_file_idx][0]
        print(f"\n СТАРТ ФОЛДА {fold+1} | Валидация на группе: {test_group}")
        tr_files = [dataset[i] for i in tr_file_idx]; va_files = [dataset[i] for i in va_file_idx]
        
        tr_det_raw_concat = np.vstack([f["data_mat"] for f in tr_files])
        scaler_det = StandardScaler()
        scaler_det.fit(tr_det_raw_concat)
        for f in tr_files + va_files: 
            f["data_mat_det_scaled"] = scaler_det.transform(f["data_mat"])
            
        X_tr_det_list, y_tr_det_list = [], []
        for f in tr_files:
            X, T, y = create_windows(f["data_mat_det_scaled"], f["time_array"], WIN_SIZE_DET, STEP_SIZE_DET, f["raw_json"], is_gait_segmenter=False)
            X_tr_det_list.append(X); y_tr_det_list.append(y)
        X_tr_det = np.vstack(X_tr_det_list)
        
        X_va_det_list, y_va_det_list = [], []
        for f in va_files:
            X, T, y = create_windows(f["data_mat_det_scaled"], f["time_array"], WIN_SIZE_DET, STEP_SIZE_DET, f["raw_json"], is_gait_segmenter=False)
            X_va_det_list.append(X); y_va_det_list.append(y)
        X_va_det = np.vstack(X_va_det_list)
        
        y_tr_det = np.nan_to_num(np.concatenate(y_tr_det_list), nan=0).astype(np.int64)
        y_va_det = np.nan_to_num(np.concatenate(y_va_det_list), nan=0).astype(np.int64)

        num_zeros = np.sum(y_tr_det == 0); num_ones = np.sum(y_tr_det == 1)
        det_class_weights = [float(num_ones / (num_zeros + 1e-6)), 1.0]
        
        det_tr_loader = DataLoader(SegDataset(X_tr_det, y_tr_det, True), batch_size=CFG["batch_size"], shuffle=True)
        det_va_loader = DataLoader(SegDataset(X_va_det, y_va_det, False), batch_size=CFG["batch_size"], shuffle=False)

        det_model = AttentionLSTM(input_dim=N_FEATURES, n_classes=2).to(DEVICE)
        det_model, det_tr_loss, det_va_loss = train_model(det_model, det_tr_loader, det_va_loader, class_weights=det_class_weights, is_detector=True)
        plot_loss_curves(det_tr_loss, det_va_loss, fold+1, "Detector")

        movement_chunks = []
        for f in tr_files:
            t_start = f["raw_json"]["segment_1"]["start_time"]
            t_end = f["raw_json"]["point_5"]["time_sec"]
            st_idx = np.searchsorted(f["time_array"], t_start)
            en_idx = np.searchsorted(f["time_array"], t_end)
            if en_idx > st_idx: movement_chunks.append(f["data_mat"][st_idx:en_idx])
                
        seg_scaler_data = np.vstack(movement_chunks) if movement_chunks else tr_det_raw_concat
        scaler_seg = StandardScaler(); scaler_seg.fit(seg_scaler_data)
        
        X_tr_seg_list, y_tr_seg_list = [], []
        for f in tr_files:
            t_start = f["raw_json"]["segment_1"]["start_time"]
            t_end = f["raw_json"]["point_5"]["time_sec"]
            st_idx = np.searchsorted(f["time_array"], t_start)
            en_idx = np.searchsorted(f["time_array"], t_end)
            if en_idx > st_idx:
                data_mat_main_scaled = scaler_seg.transform(f["data_mat"][st_idx:en_idx])
                X, T, y = create_windows(data_mat_main_scaled, f["time_array"][st_idx:en_idx], WIN_SIZE_SEG, STEP_SIZE_SEG, f["raw_json"], is_gait_segmenter=True)
                if len(X) > 0: X_tr_seg_list.append(X); y_tr_seg_list.append(y)
        
        X_va_seg_list, y_va_seg_list = [], []
        for f in va_files:
            t_start = f["raw_json"]["segment_1"]["start_time"]
            t_end = f["raw_json"]["point_5"]["time_sec"]
            st_idx = np.searchsorted(f["time_array"], t_start)
            en_idx = np.searchsorted(f["time_array"], t_end)
            if en_idx > st_idx:
                data_mat_main_scaled = scaler_seg.transform(f["data_mat"][st_idx:en_idx])
                X, T, y = create_windows(data_mat_main_scaled, f["time_array"][st_idx:en_idx], WIN_SIZE_SEG, STEP_SIZE_SEG, f["raw_json"], is_gait_segmenter=True)
                if len(X) > 0: X_va_seg_list.append(X); y_va_seg_list.append(y)

        X_tr_seg = np.vstack(X_tr_seg_list) if X_tr_seg_list else np.empty((0, WIN_SIZE_SEG, N_FEATURES))
        y_tr_seg_temp = np.concatenate(y_tr_seg_list) if y_tr_seg_list else np.array([])
        X_va_seg = np.vstack(X_va_seg_list) if X_va_seg_list else np.empty((0, WIN_SIZE_SEG, N_FEATURES))
        y_va_seg_temp = np.concatenate(y_va_seg_list) if y_va_seg_list else np.array([])
        
        tr_seg_mask = (y_tr_seg_temp == 0) | (y_tr_seg_temp == 1)
        y_tr_seg = np.where((y_tr_seg_temp == 0) | (y_tr_seg_temp == 1), y_tr_seg_temp, -1).astype(np.int64)
        y_va_seg = np.where((y_va_seg_temp == 0) | (y_va_seg_temp == 1), y_va_seg_temp, -1).astype(np.int64)
        va_seg_mask = (y_va_seg == 0) | (y_va_seg == 1)
        
        X_tr_seg_filtered = X_tr_seg[tr_seg_mask] if len(X_tr_seg) > 0 else np.array([])
        y_tr_seg_filtered = y_tr_seg[tr_seg_mask] if len(y_tr_seg) > 0 else np.array([])
        X_va_seg_filtered = X_va_seg[va_seg_mask] if len(X_va_seg) > 0 else np.array([])
        y_va_seg_filtered = y_va_seg[va_seg_mask] if len(y_va_seg) > 0 else np.array([])

        num_zeros_seg = np.sum(y_tr_seg_filtered == 0) if len(y_tr_seg_filtered) > 0 else 0
        num_ones_seg = np.sum(y_tr_seg_filtered == 1) if len(y_tr_seg_filtered) > 0 else 0
        seg_class_weights = [float(num_ones_seg / (num_zeros_seg + 1e-6)), 1.0]
        
        has_seg_train_data = len(X_tr_seg_filtered) > 0 and len(y_tr_seg_filtered) > 0
        has_seg_val_data = len(X_va_seg_filtered) > 0 and len(y_va_seg_filtered) > 0

        if has_seg_train_data:
            seg_tr_loader = DataLoader(SegDataset(X_tr_seg_filtered, y_tr_seg_filtered, True), batch_size=CFG["batch_size"], shuffle=True)
            seg_va_loader = DataLoader(SegDataset(X_va_seg_filtered, y_va_seg_filtered, False), batch_size=CFG["batch_size"], shuffle=False) if has_seg_val_data else seg_tr_loader 
            seg_model = AttentionLSTM(input_dim=N_FEATURES, n_classes=2).to(DEVICE)
            seg_model, seg_tr_loss, seg_va_loss = train_model(seg_model, seg_tr_loader, seg_va_loader, class_weights=seg_class_weights, is_detector=False)
            plot_loss_curves(seg_tr_loss, seg_va_loss, fold+1, "Segmenter")
        else:
            print("Пропуск обучения сегментатора: нет данных участков движения в тренировочном фолде.")
            seg_model = None; seg_tr_loss, seg_va_loss = [], []

        f1_det_list, bal_acc_det_list = [], []
        f1_seg_list, bal_acc_seg_list = [], []
        delta_start_list, delta_end_list = [], []
        fold_det_true_accum, fold_det_pred_accum = [], []
        fold_seg_true_accum, fold_seg_pred_accum = [], []

        for f in va_files:
            X_vis_det, T_vis_det, det_gt = create_windows(f["data_mat_det_scaled"], f["time_array"], WIN_SIZE_DET, STEP_SIZE_DET, f["raw_json"], is_gait_segmenter=False)
            det_gt = np.where((det_gt == 0) | (det_gt == 1), det_gt, 0)
            
            det_model.eval()
            with torch.no_grad():
                det_pred = torch.argmax(det_model(torch.FloatTensor(X_vis_det).to(DEVICE)), dim=1).cpu().numpy()
            det_pred = smooth_predictions(det_pred, k=7)

            main_segment = find_true_movement_boundaries(det_pred, fs_windows=None, min_duration_sec=1.5)
            if main_segment is not None:
                clean_det_pred = np.zeros_like(det_pred); clean_det_pred[main_segment[0]:main_segment[1]] = 1; det_pred = clean_det_pred
            else: det_pred = np.zeros_like(det_pred)

            gt_t1 = f["raw_json"]["segment_1"]["start_time"]; gt_t5 = f["raw_json"]["point_5"]["time_sec"]
            gt_t2 = f["raw_json"]["segment_1"]["end_time"]; gt_t3 = f["raw_json"]["segment_2"]["start_time"]
            gt_t4 = gt_t5 - f["raw_json"]["point_5"]["turn2_duration_sec"]

            det_metrics = calculate_and_format_metrics(det_gt, det_pred)
            cm_det = confusion_matrix(det_gt, det_pred, labels=[0, 1])
            if cm_det.shape == (2, 2): tn_det, fp_det, fn_det, tp_det = cm_det.ravel()
            else: tn_det, fp_det, fn_det, tp_det = 0, 0, 0, 0
            
            f1_det_list.append(det_metrics["Macro F1"]); bal_acc_det_list.append(det_metrics["Balanced Accuracy"])
            fold_det_true_accum.extend(det_gt); fold_det_pred_accum.extend(det_pred)

            seg_pred_resampled = np.zeros(len(T_vis_det), dtype=int)
            fold_seg_gt = np.full(len(T_vis_det), -1, dtype=int)

            if main_segment is not None and seg_model is not None:
                h_st, h_en = main_segment
                st_time = T_vis_det[h_st]; last_det_idx = min(h_en - 1, len(T_vis_det) - 1)
                end_time_with_window = T_vis_det[last_det_idx] + WIN_SIZE_DET
                st_idx = np.searchsorted(f["time_array"], st_time)
                en_idx = min(len(f["time_array"]), np.searchsorted(f["time_array"], end_time_with_window))
                
                if en_idx > st_idx:
                    data_mat_main_scaled = scaler_seg.transform(f["data_mat"][st_idx:en_idx])
                    X_seg, T_seg, seg_gt_main = create_windows(data_mat_main_scaled, f["time_array"][st_idx:en_idx], WIN_SIZE_SEG, STEP_SIZE_SEG, f["raw_json"], is_gait_segmenter=True)
                    if len(X_seg) > 0:
                        seg_model.eval()
                        with torch.no_grad():
                            seg_part = torch.argmax(seg_model(torch.FloatTensor(X_seg).to(DEVICE)), dim=1).cpu().numpy()
                        if len(seg_part) >= 5: seg_part = medfilt(seg_part, kernel_size=5)
                        for i, val in enumerate(seg_part):
                            t_point = T_seg[i]; det_idx = np.argmin(np.abs(T_vis_det - t_point))
                            if h_st <= det_idx < h_en:
                                seg_pred_resampled[det_idx] = val; fold_seg_gt[det_idx] = seg_gt_main[i]

            valid_mask = (fold_seg_gt == 0) | (fold_seg_gt == 1)
            if np.sum(valid_mask) > 0:
                seg_metrics = calculate_and_format_metrics(fold_seg_gt[valid_mask], seg_pred_resampled[valid_mask])
                cm_seg = confusion_matrix(fold_seg_gt[valid_mask], seg_pred_resampled[valid_mask], labels=[0, 1])
                if cm_seg.shape == (2, 2): tn_seg, fp_seg, fn_seg, tp_seg = cm_seg.ravel()
                else: tn_seg, fp_seg, fn_seg, tp_seg = 0, 0, 0, 0
                f1_seg_list.append(seg_metrics["Macro F1"]); bal_acc_seg_list.append(seg_metrics["Balanced Accuracy"])
                fold_seg_true_accum.extend(fold_seg_gt[valid_mask]); fold_seg_pred_accum.extend(seg_pred_resampled[valid_mask])
            else:
                seg_metrics = {k: 0.0 for k in ["Macro F1", "Balanced F1", "Balanced Accuracy", "Macro Recall", "Sensitivity", "Specificity", "Precision_1"]}
                tn_seg, fp_seg, fn_seg, tp_seg = 0, 0, 0, 0

            point_seg_pred = convert_windows_to_points(f["df_proc"], T_vis_det, seg_pred_resampled, det_pred, main_segment)
            calculated_timestamps = extract_precise_timestamps(f["df_proc"]["time_sec"].values, point_seg_pred, det_pred, main_segment, T_vis_det)
            plot_results_v3(f["df_proc"], T_vis_det, det_pred, point_seg_pred, f["raw_json"], calculated_timestamps, f"Fold {fold+1} | {f['name']}", f"{f['name']}_predicted_timestamps.json")

            d_start = calculated_timestamps["T1_start"] - gt_t1 if calculated_timestamps["T1_start"] is not None else np.nan
            d_end = calculated_timestamps["T5_end"] - gt_t5 if calculated_timestamps["T5_end"] is not None else np.nan
            d_t2 = calculated_timestamps["T2_turn1_start"] - gt_t2 if calculated_timestamps["T2_turn1_start"] is not None else np.nan
            d_t3 = calculated_timestamps["T3_walk2_start"] - gt_t3 if calculated_timestamps["T3_walk2_start"] is not None else np.nan
            d_t4 = calculated_timestamps["T4_turn2_start"] - gt_t4 if calculated_timestamps["T4_turn2_start"] is not None else np.nan
            
            if not np.isnan(d_start): delta_start_list.append(abs(d_start))
            if not np.isnan(d_end): delta_end_list.append(abs(d_end))

            statistical_records.append({
                "File_Name": f["name"], "Group_Name": f["group_name"], "Fold": fold + 1, 
                "Delta_T1": d_start, "Delta_T2": d_t2, "Delta_T3": d_t3, "Delta_T4": d_t4, "Delta_T5": d_end,
                "Det_Macro_F1": det_metrics["Macro F1"], 
                "Det_Balanced_F1": det_metrics["Balanced F1"],
                "Det_Balanced_Accuracy": det_metrics["Balanced Accuracy"],
                "Det_Macro_Recall": det_metrics["Macro Recall"],
                "Det_Sensitivity": det_metrics["Sensitivity"],
                "Det_Specificity": det_metrics["Specificity"],
                "Det_Precision_1": det_metrics["Precision_1"],
                "Det_TP": tp_det, "Det_TN": tn_det, "Det_FP": fp_det, "Det_FN": fn_det,
                "Seg_Macro_F1": seg_metrics["Macro F1"],
                "Seg_Balanced_F1": seg_metrics["Balanced F1"],
                "Seg_Balanced_Accuracy": seg_metrics["Balanced Accuracy"],
                "Seg_Macro_Recall": seg_metrics["Macro Recall"],
                "Seg_Sensitivity": seg_metrics["Sensitivity"],
                "Seg_Specificity": seg_metrics["Specificity"],
                "Seg_Precision_1": seg_metrics["Precision_1"],
                "Seg_TP": tp_seg, "Seg_TN": tn_seg, "Seg_FP": fp_seg, "Seg_FN": fn_seg
            })
            
        if fold_det_true_accum:
            calculate_and_format_metrics(fold_det_true_accum, fold_det_pred_accum, title=f"ДЕТЕКТОР — Итоги Фолда {fold+1} [{test_group}]", print_table=True)
            all_det_true.extend(fold_det_true_accum); all_det_pred.extend(fold_det_pred_accum)
        if fold_seg_true_accum:
            calculate_and_format_metrics(fold_seg_true_accum, fold_seg_pred_accum, title=f"СЕГМЕНТАТОР — Итоги Фолда {fold+1} [{test_group}]", print_table=True)
            all_seg_true.extend(fold_seg_true_accum); all_seg_pred.extend(fold_seg_pred_accum)

        fold_metrics.append({
            "Fold": fold + 1, "Left_Out_Group": test_group,
            "Fold_Det_Macro_F1": np.mean(f1_det_list) if f1_det_list else 0.0,
            "Fold_Det_Balanced_Acc": np.mean(bal_acc_det_list) if bal_acc_det_list else 0.0,
            "Fold_Seg_Macro_F1": np.mean(f1_seg_list) if f1_seg_list else 0.0,
            "Fold_Seg_Balanced_Acc": np.mean(bal_acc_seg_list) if bal_acc_seg_list else 0.0,
            "Mean_Abs_Delta_T1": np.mean(delta_start_list) if delta_start_list else np.nan,
            "Mean_Abs_Delta_T5": np.mean(delta_end_list) if delta_end_list else np.nan
        })

    pd.DataFrame(statistical_records).to_excel("comprehensive_signals_and_deltas_report.xlsx", index=False)
    pd.DataFrame(fold_metrics).to_excel("cross_validation_folds_report.xlsx", index=False)

    print("ОБУЧЕНИЕ ФИНАЛЬНЫХ МОДЕЛЕЙ НА ВСЕХ ДАННЫХ")

    all_data_mat = np.vstack([f["data_mat"] for f in dataset])
    final_scaler_det = StandardScaler(); 
    final_scaler_det.fit(all_data_mat)
    
    X_all_det, y_all_det = [], []
    for f in dataset:
        scaled_data = final_scaler_det.transform(f["data_mat"])
        X, _, y = create_windows(scaled_data, f["time_array"], WIN_SIZE_DET, STEP_SIZE_DET, f["raw_json"], is_gait_segmenter=False)
        X_all_det.append(X); y_all_det.append(y)
        
    X_all_det = np.vstack(X_all_det)
    y_all_det = np.nan_to_num(np.concatenate(y_all_det), nan=0).astype(np.int64)
    num_zeros = np.sum(y_all_det == 0); num_ones = np.sum(y_all_det == 1)
    det_class_weights = [float(num_ones / (num_zeros + 1e-6)), 1.0]
    
    final_det_loader = DataLoader(SegDataset(X_all_det, y_all_det, True), batch_size=CFG["batch_size"], shuffle=True)
    final_det_model = AttentionLSTM(input_dim=N_FEATURES, n_classes=2).to(DEVICE)
    final_det_model, _, _ = train_model(final_det_model, final_det_loader, final_det_loader, class_weights=det_class_weights, is_detector=True)
    
    torch.save(final_det_model.state_dict(), "final_detector_model.pth")
    joblib.dump(final_scaler_det, "final_scaler_det.pkl")
    print("Финальный Детектор сохранен: final_detector_model.pth, final_scaler_det.pkl")
    
    movement_chunks_all = []
    for f in dataset:
        t_start = f["raw_json"]["segment_1"]["start_time"]
        t_end = f["raw_json"]["point_5"]["time_sec"]
        st_idx = np.searchsorted(f["time_array"], t_start)
        en_idx = np.searchsorted(f["time_array"], t_end)
        if en_idx > st_idx: movement_chunks_all.append(f["data_mat"][st_idx:en_idx])
            
    seg_scaler_data_all = np.vstack(movement_chunks_all) if movement_chunks_all else all_data_mat
    final_scaler_seg = StandardScaler(); final_scaler_seg.fit(seg_scaler_data_all)
    
    X_all_seg, y_all_seg = [], []
    for f in dataset:
        t_start = f["raw_json"]["segment_1"]["start_time"]
        t_end = f["raw_json"]["point_5"]["time_sec"]
        st_idx = np.searchsorted(f["time_array"], t_start)
        en_idx = np.searchsorted(f["time_array"], t_end)
        if en_idx > st_idx:
            data_mat_main_scaled = final_scaler_seg.transform(f["data_mat"][st_idx:en_idx])
            X, _, y = create_windows(data_mat_main_scaled, f["time_array"][st_idx:en_idx], WIN_SIZE_SEG, STEP_SIZE_SEG, f["raw_json"], is_gait_segmenter=True)
            if len(X) > 0: X_all_seg.append(X); y_all_seg.append(y)
                
    if X_all_seg:
        X_all_seg = np.vstack(X_all_seg); y_all_seg = np.concatenate(y_all_seg)
        seg_mask = (y_all_seg == 0) | (y_all_seg == 1)
        X_all_seg = X_all_seg[seg_mask]; y_all_seg = y_all_seg[seg_mask].astype(np.int64)
        
        num_zeros_seg = np.sum(y_all_seg == 0); num_ones_seg = np.sum(y_all_seg == 1)
        seg_class_weights = [float(num_ones_seg / (num_zeros_seg + 1e-6)), 1.0]
        
        final_seg_loader = DataLoader(SegDataset(X_all_seg, y_all_seg, True), batch_size=CFG["batch_size"], shuffle=True)
        final_seg_model = AttentionLSTM(input_dim=N_FEATURES, n_classes=2).to(DEVICE)
        final_seg_model, _, _ = train_model(final_seg_model, final_seg_loader, final_seg_loader, class_weights=seg_class_weights, is_detector=False)
        
        torch.save(final_seg_model.state_dict(), "final_segmenter_model.pth")
        joblib.dump(final_scaler_seg, "final_scaler_seg.pkl")
        print("Финальный Сегментатор сохранен: final_segmenter_model.pth, final_scaler_seg.pkl")
    else:
        print("Не удалось собрать данные для финального Сегментатора.")

    if len(all_det_true) > 0:
        final_det_metrics = calculate_and_format_metrics(all_det_true, all_det_pred, title="ДЕТЕКТОР — ГЛОБАЛЬНЫЕ МЕТРИКИ", print_table=True)
    else: final_det_metrics = {"Macro F1": 0.0, "Balanced Accuracy": 0.0}

    if len(all_seg_true) > 0:
        final_seg_metrics = calculate_and_format_metrics(all_seg_true, all_seg_pred, title="СЕГМЕНТАТОР — ГЛОБАЛЬНЫЕ МЕТРИКИ", print_table=True)
    else: final_seg_metrics = {"Macro F1": 0.0, "Balanced Accuracy": 0.0}

    summary_data = {
        "Model_Version": "v2_gt_segmenter_highpass_yaw",
        "Det_Global_Macro_F1": final_det_metrics.get("Macro F1", 0.0),
        "Det_Global_Balanced_Acc": final_det_metrics.get("Balanced Accuracy", 0.0),
        "Seg_Global_Macro_F1": final_seg_metrics.get("Macro F1", 0.0),
        "Seg_Global_Balanced_Acc": final_seg_metrics.get("Balanced Accuracy", 0.0),
    }
    pd.DataFrame([summary_data]).to_csv("model_summary.csv", index=False)

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import torch
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix

N_FEATURES = 9 

HOLDOUT_JSON_DIR = "/kaggle/input/datasets/daryushka/test-new-136-140/test_new_136_140/JSON_5p" 
HOLDOUT_XLSX_DIR = "/kaggle/input/datasets/daryushka/test-new-136-140/test_new_136_140/extracted_segments"
MODEL_DIR = "/kaggle/working/"                               

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {DEVICE}")

try:
    det_model = AttentionLSTM(input_dim=N_FEATURES, n_classes=2).to(DEVICE)
    det_model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "final_detector_model.pth"), map_location=DEVICE))
    det_model.eval()
    scaler_det = joblib.load(os.path.join(MODEL_DIR, "final_scaler_det.pkl"))

    seg_model = AttentionLSTM(input_dim=N_FEATURES, n_classes=2).to(DEVICE)
    seg_model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "final_segmenter_model.pth"), map_location=DEVICE))
    seg_model.eval()
    scaler_seg = joblib.load(os.path.join(MODEL_DIR, "final_scaler_seg.pkl"))
except FileNotFoundError as e:
    print(f"Ошибка загрузки файлов: {e}")

holdout_dataset = []
json_files = glob.glob(os.path.join(HOLDOUT_JSON_DIR, "**/*.json"), recursive=True)

for jf in json_files:
    with open(jf) as f:
        data = json.load(f)
    
    if "segment_1" not in data or "segment_2" not in data or "point_5" not in data:
        continue
        
    base_name = os.path.basename(jf).replace("_segments.json", "")
    xlsx_files = glob.glob(os.path.join(HOLDOUT_XLSX_DIR, f"*{base_name}*.xlsx"))
    
    if len(xlsx_files) > 0:
        df = pd.read_excel(xlsx_files[0])
        data_mat, time_array, df_proc = resample_and_prep(df) 
        
        holdout_dataset.append({
            "name": base_name,
            "data_mat": data_mat,
            "time_array": time_array,
            "df_proc": df_proc, 
            "raw_json": data
        })

print(f"Загружено {len(holdout_dataset)} файлов для инференса.")
if len(holdout_dataset) == 0:
    raise RuntimeError("Не найдено файлов для инференса! Проверьте пути HOLDOUT_JSON_DIR и HOLDOUT_XLSX_DIR.")

all_det_true, all_det_pred = [], []
all_seg_true, all_seg_pred = [], []
statistical_records = []

for file_data in holdout_dataset:
    name = file_data["name"]
    json_data = file_data["raw_json"]
    data_mat = file_data["data_mat"]
    time_array = file_data["time_array"]
    df_proc = file_data["df_proc"] 

    scaled_data_det = scaler_det.transform(data_mat)
    X_det, T_det, y_det_gt = create_windows(scaled_data_det, time_array, WIN_SIZE_DET, STEP_SIZE_DET, json_data, is_gait_segmenter=False)
    y_det_gt = np.where((y_det_gt == 0) | (y_det_gt == 1), y_det_gt, 0)
    
    with torch.no_grad():
        det_preds = torch.argmax(det_model(torch.FloatTensor(X_det).to(DEVICE)), dim=1).cpu().numpy()
    
    det_preds_smooth = smooth_predictions(det_preds, k=7)
    main_segment = find_true_movement_boundaries(det_preds_smooth, fs_windows=None, min_duration_sec=1.5)
    
    if main_segment is not None:
        clean_det_pred = np.zeros_like(det_preds_smooth)
        clean_det_pred[main_segment[0]:main_segment[1]] = 1
        det_preds_smooth = clean_det_pred
    else:
        det_preds_smooth = np.zeros_like(det_preds_smooth)

    all_det_true.extend(y_det_gt)
    all_det_pred.extend(det_preds_smooth)

    seg_pred_resampled = np.zeros(len(T_det), dtype=int)
    fold_seg_gt = np.full(len(T_det), -1, dtype=int)
    
    if main_segment is not None:
        h_st, h_en = main_segment
        st_time = T_det[h_st]
        last_det_idx = min(h_en - 1, len(T_det) - 1)
        end_time_with_window = T_det[last_det_idx] + WIN_SIZE_DET
        
        st_idx = np.searchsorted(time_array, st_time)
        en_idx = min(len(time_array), np.searchsorted(time_array, end_time_with_window))
        
        if en_idx > st_idx:
            raw_segment_data = data_mat[st_idx:en_idx]
            segment_time_array = time_array[st_idx:en_idx]
            scaled_segment_data = scaler_seg.transform(raw_segment_data)
            
            X_seg, T_seg, y_seg_gt_main = create_windows(scaled_segment_data, segment_time_array, WIN_SIZE_SEG, STEP_SIZE_SEG, json_data, is_gait_segmenter=True)
            
            if len(X_seg) > 0:
                with torch.no_grad():
                    seg_preds = torch.argmax(seg_model(torch.FloatTensor(X_seg).to(DEVICE)), dim=1).cpu().numpy()
                
                seg_preds_smooth = smooth_predictions(seg_preds, k=5) if len(seg_preds) >= 5 else seg_preds
                
                for i, val in enumerate(seg_preds_smooth):
                    t_point = T_seg[i]
                    det_idx = np.argmin(np.abs(T_det - t_point))
                    if h_st <= det_idx < h_en:
                        seg_pred_resampled[det_idx] = val
                        fold_seg_gt[det_idx] = y_seg_gt_main[i]

    valid_mask = (fold_seg_gt == 0) | (fold_seg_gt == 1)
    if np.sum(valid_mask) > 0:
        all_seg_true.extend(fold_seg_gt[valid_mask])
        all_seg_pred.extend(seg_pred_resampled[valid_mask])

    point_seg_pred = convert_windows_to_points(df_proc, T_det, seg_pred_resampled, det_preds_smooth, main_segment)
    
    calculated_timestamps = extract_precise_timestamps(
        time_signals=df_proc["time_sec"].values, 
        point_seg_pred=point_seg_pred, 
        det_pred=det_preds_smooth, 
        pred_segment=main_segment, 
        T_det=T_det
    )

    plot_results_v3(
        df=df_proc, 
        T=T_det, 
        det_pred=det_preds_smooth, 
        point_seg_pred=point_seg_pred, 
        json_data=json_data, 
        timestamps=calculated_timestamps,
        title=f"Holdout Inference | {name}", 
        save_json_name=f"{name}_holdout_timestamps.json"
    )
    
    gt_t1 = json_data["segment_1"]["start_time"]
    gt_t5 = json_data["point_5"]["time_sec"]
    
    d_start = calculated_timestamps["T1_start"] - gt_t1 if calculated_timestamps["T1_start"] is not None else np.nan
    d_end = calculated_timestamps["T5_end"] - gt_t5 if calculated_timestamps["T5_end"] is not None else np.nan
    
    statistical_records.append({
        "File_Name": name,
        "Delta_T1_sec": d_start,
        "Delta_T5_sec": d_end,
        "Det_Found_Movement": main_segment is not None,
        "T1_Predicted": calculated_timestamps["T1_start"],
        "T5_Predicted": calculated_timestamps["T5_end"]
    })


print("МЕТРИКИ НА ОТЛОЖЕННОЙ ВЫБОРКЕ")


if len(all_det_true) > 0:
    calculate_and_format_metrics(all_det_true, all_det_pred, title="ДЕТЕКТОР (Holdout Set)", print_table=True)

if len(all_seg_true) > 0:
    calculate_and_format_metrics(all_seg_true, all_seg_pred, title="СЕГМЕНТАТОР (Holdout Set)", print_table=True)

df_stats = pd.DataFrame(statistical_records)

print(f"Средняя абсолютная ошибка T1 (начало): {df_stats['Delta_T1_sec'].abs().mean():.2f} сек")
print(f"Средняя абсолютная ошибка T5 (конец):  {df_stats['Delta_T5_sec'].abs().mean():.2f} сек")
print(f"Доля файлов, где движение найдено: {df_stats['Det_Found_Movement'].mean()*100:.1f}%")

report_path = "holdout_inference_report.csv"
df_stats.to_csv(report_path, index=False)
print(f"\nДетальный отчет по каждому файлу сохранен в: {report_path}")